In [1]:
# Section A. Model Training
# =============================================================================

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# A.1 Load the curated dataset (produced by 01_ingest_clean.ipynb)
# ----------------------------------------------------------------------------
print("Loading curated dataset...")
df = pd.read_parquet("../data/curated/home_credit_curated.parquet")

print(f"Dataset shape: {df.shape}")
print(f"TARGET distribution:\n{df['TARGET'].value_counts(normalize=True).round(4)}")
print(f"Columns: {list(df.columns)}")

# ----------------------------------------------------------------------------
# Separate features and target
# ----------------------------------------------------------------------------
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

# ----------------------------------------------------------------------------
# A.1 Train-test split (80/20, stratified by TARGET)
# ----------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train set shape: {X_train.shape} | Test set shape: {X_test.shape}")
print(f"Train TARGET ratio: {y_train.mean():.4f} | Test TARGET ratio: {y_test.mean():.4f}")

Loading curated dataset...
Dataset shape: (250655, 176)
TARGET distribution:
TARGET
0    0.9132
1    0.0868
Name: proportion, dtype: float64
Columns: ['SK_ID_CURR', 'TARGET', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_CHANGE', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG

In [2]:
# ----------------------------------------------------------------------------
# A.2 Address class imbalance
# ----------------------------------------------------------------------------
# XGBoost: use scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"XGBoost scale_pos_weight (neg/pos) = {scale_pos_weight:.2f}")

# Logistic Regression: will use class_weight='balanced' + feature scaling

XGBoost scale_pos_weight (neg/pos) = 10.52


In [3]:
# ----------------------------------------------------------------------------
# A.3 Train at least two models
#    • Gradient boosted trees → XGBoost (required)
#    • Second model → Logistic Regression (with scaling)
# ----------------------------------------------------------------------------

print("\n=== Training XGBoost ===")
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=42,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
print("XGBoost training completed.")

print("\n=== Training Logistic Regression ===")
# Scale features (LR is sensitive to scale)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
)
lr_model.fit(X_train_scaled, y_train)
print("Logistic Regression training completed.")


=== Training XGBoost ===
XGBoost training completed.

=== Training Logistic Regression ===
Logistic Regression training completed.


In [4]:
# ----------------------------------------------------------------------------
# A.4 Compute metrics on the test set for every model
#    Metrics required: ROC-AUC, MCC, precision, recall, F1-score
# ----------------------------------------------------------------------------

def get_metrics(model, X, y_true, model_name, is_scaled=False):
    if is_scaled:
        y_pred_proba = model.predict_proba(X)[:, 1]
        y_pred = model.predict(X)
    else:
        y_pred_proba = model.predict_proba(X)[:, 1]
        y_pred = model.predict(X)

    return {
        "Model": model_name,
        "ROC-AUC": round(roc_auc_score(y_true, y_pred_proba), 4),
        "MCC": round(matthews_corrcoef(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-Score": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }


# Evaluate both models
metrics_list = []
metrics_list.append(get_metrics(xgb_model, X_test, y_test, "XGBoost"))
metrics_list.append(
    get_metrics(lr_model, X_test_scaled, y_test, "Logistic Regression", is_scaled=True)
)

# ----------------------------------------------------------------------------
# Create and display comparison table
# ----------------------------------------------------------------------------
comparison_df = pd.DataFrame(metrics_list)
print("\n" + "=" * 60)
print("MODEL PERFORMANCE COMPARISON ON TEST SET")
print("=" * 60)
print(comparison_df.to_string(index=False))

# ----------------------------------------------------------------------------
# Save the comparison table (for later use in the report)
# ----------------------------------------------------------------------------
comparison_df.to_csv("../data/curated/model_comparison.csv", index=False)
print("\nComparison table saved to ../data/curated/model_comparison.csv")

# Optional: save models for later sections (SHAP, DiCE, etc.)
#import joblib
#joblib.dump(xgb_model, "../models/xgb_model.pkl")
#joblib.dump(lr_model, "../models/lr_model.pkl")
#joblib.dump(scaler, "../models/lr_scaler.pkl")
#print("Models and scaler saved to ../models/")

#print("\nSection A completed. Proceed to Section B (Impact Simulation).")


MODEL PERFORMANCE COMPARISON ON TEST SET
              Model  ROC-AUC    MCC  Precision  Recall  F1-Score
            XGBoost   0.7555 0.2300     0.1869  0.6353    0.2888
Logistic Regression   0.7430 0.2152     0.1699  0.6795    0.2718

Comparison table saved to ../data/curated/model_comparison.csv
